# 21 AG ledger, stage 1: the six P50 SMR mandate runs against the no-mandate baseline

**Purpose.** To measure who gains and who loses when the mandate builds SMRs, this notebook builds
the paper-AG ledger for stage 1 of `docs/protocol.md` (signed 2026-10-28; amendments A1–A8 in §11).
Each ledger row is one actor's money flow or one emissions change, measured as policy run minus
baseline. The policy runs are `test1_smr100_{t}_p50` for the six cost targets; the baseline is
`itcfb_fbR_none` (protocol §2).

**What it writes** (`results/exports/21/`):
- `ag_actor_pv.csv`: every actor entry as a present value at every rate, per horizon, support form
  and funding case (2025 $B);
- `ag_views.csv`: both resident views for every variant of protocol §5, with check C3;
- `ag_annual_A.csv`: every actor entry by year under horizon A (2025 $B per year);
- `ag_emissions_delta.csv`: emissions changes by gas and year (tonnes);
- `ag_closure.csv`: the residuals of checks C1, C2, the RPS account and C3, with both denominators
  and the 0.5% flags (protocol §6);
- `ag_c4_support.csv`, `ag_c5_firstcut.csv`: checks C4 and C5.

**What it does not do.** It states no hypothesis and picks no headline variant. Patterns go to
`docs/observations.md` by hand (protocol §10). It opens no ReEDS outputs file: every input is an
extract from notebook 20 or a file in `data/external`.


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths, plotstyle as ps
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("21")
TO25 = ps.to2025(2004)
print(f"exports to {paths.rel(OUT)}; 2004 $ -> 2025 $ x {TO25:.4f}")

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.
exports to z-ethan\results\exports\21; 2004 $ -> 2025 $ x 1.7021


## 1. Inputs and the pairs

`load_external` reads the gas quantities (AEO 2026, amendment A3), the carbon values (EPA 2023 global
at 1.5/2.0/2.5% and EPA 2019 domestic at 3/7%, all in 2004 $ per tonne) and the CACES marginal damages
(z90 zone means, as ReEDS's `health_damage_calculations.py` computes them).

In [2]:
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
BASE = ("itcfb", "fbR_none")
PAIRS = {t: (("smr100", f"smr100_{t}_p50"), BASE) for t in TARGETS}
EXT = L.load_external()
CRF = L.crf_after_2026()
print(f"CRF after 2026 = {CRF}; 1/CRF = {1 / CRF:.4f}")
q = pd.DataFrame({"Q_np (quads)": EXT["Q_np"] / 1e9, "Q_exp (quads)": EXT["Q_exp"] / 1e9}).loc[[2025, 2030, 2040, 2050]]
print(q.round(2).to_string())
print("carbon values in 2030, 2004 $/t CO2:",
      {b: round(float(L.series_on([2030], EXT["scc"][(b, "CO2")])[0]), 1) for b in L.SCC_PAIRS})
md_ = EXT["md"]
print("CACES:", md_["variant"].nunique(), "variants,", md_["r"].nunique(), "zones; pollutants", sorted(md_["pollutant"].unique()))

CRF after 2026 = 0.06773; 1/CRF = 14.7645
      Q_np (quads)  Q_exp (quads)
2025         17.60           5.92
2030         16.98           9.53
2040         17.53          12.49
2050         18.45          13.56
carbon values in 2030, 2004 $/t CO2: {'global EPA2023 1.5%': 280.4, 'global EPA2023 2.0%': 167.9, 'global EPA2023 2.5%': 105.1, 'domestic EPA2019 3%': 6.3, 'domestic EPA2019 7%': 0.8}
CACES: 6 variants, 89 zones; pollutants ['NH3', 'NOX', 'PM25', 'SO2', 'VOC']


## 2. Checks on the inputs before any ledger

1. **Baseline.** `fbR_none` builds no new SMR and no endogenous large reactor in 2026–2050. Its new
   large-reactor MW must equal the exogenous units in ReEDS's generator database (amendment A9), and
   each policy run must carry the same large-reactor MW.
2. **Solve years and dropped load.** Both runs of each pair share solve years; dropped load is zero.
3. **H2 overlap (protocol §9 item 7).** Q_np includes the AEO hydrogen-production gas (row 3c). That is
   allowed only if ReEDS's own gas quantity holds no gas for hydrogen. ReEDS produces H2 only in
   electrolyzers when no steam-methane-reforming technology (`smr`, `smr_ccs`) appears in any
   capacity or production-load key.
4. **ITC sign (amendment A4).** `tax_expenditure_itc` equals the ITC rows of `systemcost`; positive
   values are listed by technology.
5. **Gas-curve fuel.** The objective fuel cost of the gas-curve technologies sums to `gascost_cendiv`.
6. **Group map.** The technologies that fall into "other" are listed.

In [3]:
rd = lambda b, k: pd.read_parquet(L.EXTRACTS_AG / b / f"{k}.parquet")
cn = rd("itcfb", "cap_new_ivrt")
# exogenous large-reactor units in ReEDS's generator database that start after the 2023 solve year
# (Vogtle 4 and three restarts); every run carries them (protocol amendment A9)
gdb = pd.read_csv(paths.REPO / "inputs" / "capacity_exogenous" / "ReEDS_generator_database_final_EIA-NEMS.csv",
                  usecols=["tech", "T_PNM", "StartYear", "summer_power_capacity_MW"], low_memory=False)
exo = gdb[(gdb["tech"] == "nuclear") & (gdb["StartYear"] > 2023)]
print("exogenous nuclear units after 2023:")
print(exo.groupby(["T_PNM", "StartYear"])["summer_power_capacity_MW"].sum().to_string())
b_new = cn[(cn["case"] == BASE[1]) & (cn["t"].astype(int) >= 2026)]
smr_mw = float(b_new.loc[b_new["i"] == "nuclear-smr", "Value"].sum())
lr_mw = float(b_new.loc[b_new["i"] == "nuclear", "Value"].sum())
print(f"1. baseline 2026-2050: new SMR {smr_mw:.3f} MW; new large reactor {lr_mw:.1f} MW; "
      f"exogenous units {exo['summer_power_capacity_MW'].sum():.1f} MW")
assert smr_mw < 1e-6
assert abs(lr_mw - exo["summer_power_capacity_MW"].sum()) < 0.5, "endogenous new large reactors in the baseline"
for t in TARGETS:
    c2 = rd("smr100", "cap_new_ivrt"); c2 = c2[(c2["case"] == f"smr100_{t}_p50") & (c2["t"].astype(int) >= 2026) & (c2["i"] == "nuclear")]
    assert abs(c2["Value"].sum() - lr_mw) < 0.5, f"{t}: large-reactor builds differ from the baseline"
print("   the six policy runs carry the same large-reactor MW")

for b in ("smr100", "itcfb"):
    dl = rd(b, "dropped_load")
    dl = dl[dl["t"].astype(int) >= 2026]
    print(f"2. dropped load 2026+ in {b} (MWh, max over runs):", float(dl["Value"].abs().max()) if len(dl) else 0.0)

techs = set()
for b in ("smr100", "itcfb"):
    for k in ("cap_nat", "prod_load_ann", "cap_new_ivrt"):
        techs |= set(rd(b, k)["i"].unique())
smr_h2 = sorted(i for i in techs if i.lower().startswith("smr"))
h2_techs = sorted(rd("smr100", "prod_load_ann")["i"].unique())
print("3. steam-methane-reforming techs found:", smr_h2, "| production-load techs:", h2_techs)
assert not smr_h2 and set(h2_techs) <= {"electrolyzer"} | {i for i in h2_techs if i.lower().startswith("dac")}

for b, c in [PAIRS["eo"][0], BASE]:
    sc = rd(b, "systemcost"); sc = sc[sc["case"] == c]
    itc = sc[sc["sys_costs"].str.match(L.ITC)].groupby("t")["Value"].sum()
    te = rd(b, "tax_expenditure_itc"); te = te[te["case"] == c].set_index("t")["Value"]
    gap = (itc.reindex(te.index.union(itc.index)).fillna(0) - te.reindex(te.index.union(itc.index)).fillna(0)).abs().max()
    tb = rd(b, "systemcost_techba"); tb = tb[(tb["case"] == c) & tb["sys_costs"].str.match(L.ITC) & (tb["Value"] > 0)]
    pos = tb.groupby("i")["Value"].sum().sort_values(ascending=False) / 1e6
    print(f"4. {c}: max |ITC rows - tax_expenditure_itc| = {gap:.3g} $; positive ITC rows by tech (2004 $M, all years):",
          pos.round(2).to_dict())

for b, c in [PAIRS["eo"][0], BASE]:
    tb = rd(b, "systemcost_techba"); tb = tb[(tb["case"] == c) & (tb["sys_costs"] == "op_fuelcosts_objfn")]
    f = tb[tb["i"].map(L.on_gas_curve)].groupby("t")["Value"].sum()
    g = rd(b, "gascost_cendiv"); g = g[g["case"] == c].groupby("t")["Value"].sum()
    d = (f - g).loc[2026:]
    print(f"5. {c}: max |gas-curve fuel - gascost_cendiv| 2026+ = {d.abs().max() / 1e9:.4f} $B")

other = sorted(i for i in techs if L.group(i) == "other")
print("6. technologies in 'other':", other)

exogenous nuclear units after 2023:
T_PNM                       StartYear
Duane Arnold Energy Center  2029          601.4
Palisades                   2026          768.5
Three Mile Island           2027          802.8
Vogtle                      2024         1114.0
1. baseline 2026-2050: new SMR 0.000 MW; new large reactor 3286.7 MW; exogenous units 3286.7 MW
   the six policy runs carry the same large-reactor MW
2. dropped load 2026+ in smr100 (MWh, max over runs): 0.0
2. dropped load 2026+ in itcfb (MWh, max over runs): 0.0
3. steam-methane-reforming techs found: [] | production-load techs: ['electrolyzer']


4. smr100_eo_p50: max |ITC rows - tax_expenditure_itc| = 0 $; positive ITC rows by tech (2004 $M, all years): {'wind-ofs_4': 4.400000095367432, 'wind-ofs_10': 3.390000104904175, 'wind-ofs_5': 1.6799999475479126, 'wind-ofs_8': 1.5399999618530273, 'wind-ofs_3': 1.0199999809265137, 'wind-ofs_7': 0.6000000238418579, 'wind-ofs_6': 0.5400000214576721, 'wind-ofs_1': 0.36000001430511475, 'wind-ofs_2': 0.15000000596046448, 'hydNPND': 0.10999999940395355, 'hydUND': 0.03999999910593033, 'hydUD': 0.03999999910593033, 'hydND': 0.019999999552965164}
4. fbR_none: max |ITC rows - tax_expenditure_itc| = 0 $; positive ITC rows by tech (2004 $M, all years): {'wind-ofs_4': 4.400000095367432, 'wind-ofs_10': 3.609999895095825, 'wind-ofs_5': 1.6799999475479126, 'wind-ofs_8': 1.5399999618530273, 'wind-ofs_3': 1.0199999809265137, 'wind-ofs_7': 0.6000000238418579, 'wind-ofs_6': 0.4000000059604645, 'wind-ofs_1': 0.36000001430511475, 'wind-ofs_2': 0.15000000596046448, 'hydNPND': 0.10999999940395355, 'hydUND': 0.0

## 3. The ledger of each pair

`L.pair` builds the accounts of both runs, differences them, and lays every entry out on the years of
horizon A (2026–2050, lumps as 30 CRF payments from the year after the solve year) and horizon B
(2026–2080, the 2050 values held flat, lumps whole). It values the emissions changes at each carbon
value and each CACES variant.

In [4]:
OUTS = {t: L.pair(r, b, EXT, CRF) for t, (r, b) in PAIRS.items()}
print("pairs built:", list(OUTS))

pairs built: ['eia', 'aj', 'iaea', 'mck', 'cop28', 'eo']


## 4. Closure checks (protocol §6)

**C1, C2 and the RPS account, by run and year.** The C1 and C2 residuals are reported net of the VSC
congestion rent, because the ledger gives that rent to the network. Each residual has two
denominators: its gross level (P_E, P_C or P_R in the run or the baseline, whichever gives the larger
share) and the policy-minus-baseline change that the ledger reports for that payment. Each share
carries a flag above 0.5%. A flag is a warning, not a pass/fail line (Ethan, 2026-10-28).

In [5]:
FLAG = 0.005
rows = []
for t, o in OUTS.items():
    ck = o["checks"].set_index(["who", "t"])
    for name, res, gross in [("C1 energy", lambda w: ck.loc[w, "C1_residual"] - ck.loc[w, "C1_vsc_rent"], "C1_gross"),
                             ("C2 capacity", lambda w: ck.loc[w, "C2_residual"] - ck.loc[w, "C2_vsc_rent"], "C2_gross"),
                             ("RPS account", lambda w: ck.loc[w, "RPS_residual"], "RPS_gross")]:
        r_run, r_base = res("run"), res("base")
        g_run, g_base = ck.loc["run", gross], ck.loc["base", gross]
        d_rep = g_run - g_base
        share_g = np.maximum((r_run / g_run.where(g_run != 0)).abs().fillna(0), (r_base / g_base.where(g_base != 0)).abs().fillna(0))
        share_d = ((r_run - r_base) / d_rep.where(d_rep != 0)).abs()
        rows.append(pd.DataFrame({"check": name, "target": t, "t": r_run.index, "residual_run_B": r_run.values / 1e9,
                                  "residual_base_B": r_base.values / 1e9, "residual_delta_B": (r_run - r_base).values / 1e9,
                                  "gross_run_B": g_run.values / 1e9, "delta_reported_B": d_rep.values / 1e9,
                                  "share_of_gross": share_g.values, "share_of_delta": share_d.values}))
CL = pd.concat(rows, ignore_index=True)
CL["flag_gross"] = CL["share_of_gross"] > FLAG
CL["flag_delta"] = CL["share_of_delta"] > FLAG
CL["dollars"] = "2004 $B per year"
print(CL.groupby("check")[["share_of_gross", "share_of_delta"]].max().round(4))
print("\nflag counts (of", CL.groupby("check").size().iloc[0], "run-years per check):")
print(CL.groupby("check")[["flag_gross", "flag_delta"]].sum())

             share_of_gross  share_of_delta
check                                      
C1 energy            0.0002          0.0142
C2 capacity          0.0001          0.0140
RPS account          0.4889          1.7143



flag counts (of 72 run-years per check):
             flag_gross  flag_delta
check                              
C1 energy             0           2
C2 capacity           0           3
RPS account          72          52


**The RPS account, decomposed into named terms** (amendment A5; `aglib/reduce.py` docstring):
REC trade wedge (RECs × buyer-state price − RECs × seller-state price) + reported requirement minus RECs bought (after ACP; negative when exports to Canada consume CES credits) − the reporting difference between RPS revenue and RECs × seller price (about zero). Bundled RECs are priced at the category they count toward (`docs/rps_residual_note.md`).
The last column must be zero.

In [6]:
rps = []
for t, o in OUTS.items():
    ck = o["checks"]
    x = ck[ck["who"] == "run"].set_index("t")
    rps.append(pd.DataFrame({"target": t, "t": x.index, "RPS_residual": x["RPS_residual"] / 1e9,
                             "trade_wedge": x["RPS_trade_rent"] / 1e9,
                             "requirement_minus_RECs_bought": x["RPS_requirement_minus_RECs_paid"] / 1e9,
                             "minus_reporting_difference": -x["RPS_revenue_minus_RECs_recv"] / 1e9}))
RPS = pd.concat(rps, ignore_index=True)
RPS["unexplained"] = RPS["RPS_residual"] - RPS["trade_wedge"] - RPS["requirement_minus_RECs_bought"] - RPS["minus_reporting_difference"]
print("max |unexplained| (2004 $B):", float(RPS["unexplained"].abs().max()))
print(RPS[RPS["target"] == "eo"].round(3).to_string(index=False))

max |unexplained| (2004 $B): 1.574692385930554e-07
target    t  RPS_residual  trade_wedge  requirement_minus_RECs_bought  minus_reporting_difference  unexplained
    eo 2026         4.665        5.001                         -0.336                      -0.000          0.0
    eo 2029         0.462        0.654                         -0.192                      -0.000         -0.0
    eo 2031         0.470        0.597                         -0.126                      -0.000          0.0
    eo 2032         0.255        0.340                         -0.084                      -0.000         -0.0
    eo 2033         0.234        0.287                         -0.052                      -0.001          0.0
    eo 2034         0.184        0.264                         -0.079                      -0.002         -0.0
    eo 2035         0.353        0.407                         -0.052                      -0.002         -0.0
    eo 2038         0.515        0.614                       

**C3 (views identity, amendment A5).** View 2 − View 1 − (1 − φ)·ΔΠ must equal the Δ of the market
residuals above (C1 and C2 net of the VSC rent, and the RPS account). The residual does not depend
on φ, the funding case, the support form, the MCPF or the health model; it is shown per horizon and
rate. Its denominators are |View 1| and |(1 − φ)ΔΠ| in the reference variant (funding T, lump support,
AP2 ACS, MCPF 1, φ = 0).

In [7]:
VIEWS = {t: L.views(o, True) for t, o in OUTS.items()}
rows = []
for t, o in OUTS.items():
    ck = o["checks"].set_index(["who", "t"])
    dterm = lambda c: ck.loc["run", c] - ck.loc["base", c]
    named = {"C1 net": dterm("C1_residual") - dterm("C1_vsc_rent"),
             "C2 net": dterm("C2_residual") - dterm("C2_vsc_rent"),
             "RPS": dterm("RPS_residual")}
    V = VIEWS[t]
    for (h, rk), v in V.groupby(["horizon", "rate"]):
        ref = v[(v["funding"] == "T") & (v["support_form"] == "lump") & (v["health_model"] == "AP2 ACS")
                & (v["mcpf"] == 1.0) & (v["phi"] == 0.0) & v["allowance_state"].isin(["free", "none"])].iloc[0]
        terms = {k: L.pv(L.flow(s, h), h, L.RATES[rk]) for k, s in named.items()}
        rows.append(dict(check="C3 views", target=t, horizon=h, rate=rk,
                         residual_min=v["C3_residual"].min(), residual_max=v["C3_residual"].max(),
                         **{f"term_{k}": x for k, x in terms.items()},
                         unexplained=ref["C3_residual"] - sum(terms.values()),
                         share_of_view1=abs(ref["C3_residual"] / ref["view1"]),
                         share_of_profits=abs(ref["C3_residual"] / ref["profits_domestic"])))
C3 = pd.DataFrame(rows)
C3["flag_view1"] = C3["share_of_view1"] > FLAG
C3["flag_profits"] = C3["share_of_profits"] > FLAG
C3["dollars"] = "2025 $B, PV to 2026"
print("max |unexplained| (2025 $B):", float(C3["unexplained"].abs().max()),
      "| spread of C3 across variants within a horizon-rate cell:", float((C3["residual_max"] - C3["residual_min"]).abs().max()))
print(C3.drop(columns=["dollars"]).round(4).to_string(index=False))

max |unexplained| (2025 $B): 0.0002298671451885781 | spread of C3 across variants within a horizon-rate cell: 4.6090689185263045e-06
   check target horizon rate  residual_min  residual_max  term_C1 net  term_C2 net  term_RPS  unexplained  share_of_view1  share_of_profits  flag_view1  flag_profits
C3 views    eia       A 1.5%        0.2730        0.2730       0.0088       0.0050    0.2592      -0.0000          0.0038            0.0055       False          True
C3 views    eia       A 2.0%        0.2454        0.2454       0.0081       0.0046    0.2327      -0.0000          0.0059            0.0053        True          True
C3 views    eia       A 2.5%        0.2207        0.2207       0.0075       0.0041    0.2091      -0.0000          0.0086            0.0051        True          True
C3 views    eia       A   3%        0.1987        0.1987       0.0070       0.0038    0.1880      -0.0000          0.2050            0.0050        True         False
C3 views    eia       A   7%        0

**C4 (support, amendment A2).** S per MW = shadow price × `pvf_onm` in each solve year, against
`S_t_2004_per_MW` in paper 1's `results/exports/07/t09_required_itc.csv`.

In [8]:
t09 = pd.read_csv(paths.RESULTS / "exports" / "07" / "t09_required_itc.csv")
rows = []
for t, o in OUTS.items():
    m = o["memo_run"]
    s = (m["dual"] * m["pvf"]).rename("S_ag")
    p1 = t09[t09["case"] == f"smr100_{t}_p50"].set_index("t")["S_t_2004_per_MW"].rename("S_paper1")
    j = pd.concat([s, p1], axis=1).dropna()
    j = j[j["S_paper1"] > 0]
    rows.append(j.assign(target=t, rel_diff=(j["S_ag"] / j["S_paper1"] - 1)).reset_index(names="t"))
C4 = pd.concat(rows, ignore_index=True)
print(f"C4: {len(C4)} run-years; max |relative difference| = {C4['rel_diff'].abs().max():.2e}")

C4: 55 run-years; max |relative difference| = 2.27e-07


**C5 (first-cut regression).** The first cut (`scratch/ag_firstcut_20261020/size_transfer.py`) used
different definitions: capacity payments included `res_marg_ann`, system cost included the credit rows,
lumps were discounted whole while flows stopped in 2050, and its constants were DR = 1.0536 and
2025 $ = 1.6590 × 1.0260. This cell recomputes its columns from the AG extracts with those definitions
and compares them with `size_transfer_p50.csv`.

In [9]:
FC_TO25, FC_DR, FC_PVF = 1.6590 * 1.0260, 1.0536, 14.7645
YRS = np.arange(2026, 2051); DISC = FC_DR ** -(YRS - 2026)

def fc_table(batch, case):
    years = OUTS["eo"]["years"]
    der = pd.read_parquet(L.EXTRACTS_AG / batch / "derived_ledger.parquet")
    der = der[der["case"] == case].pivot_table(index="t", columns="item", values="Value").reindex(years).fillna(0)
    sc = rd(batch, "systemcost"); sc = sc[sc["case"] == case]
    g = lambda d: d.groupby("t")["Value"].sum().reindex(years).fillna(0)
    em = rd(batch, "emit_nat"); em = em[(em["case"] == case) & (em["eall"] == "CO2") & (em["etype"] == "process")]
    cap = rd(batch, "cap_nat"); cap = cap[(cap["case"] == case) & (cap["i"] == "nuclear-smr")]
    cn = rd(batch, "cap_new_ivrt"); cn = cn[(cn["case"] == case) & (cn["i"] == "nuclear-smr")]
    du = rd(batch, "nuclear_cap_price") if (L.EXTRACTS_AG / batch / "nuclear_cap_price.parquet").exists() else None
    D = pd.DataFrame({"load": der["P_E"], "res_marg": der["P_C"] + der.get("P_C_ann_firstcut", 0), "state_rps": der["P_R"],
                      "load_MWh": der["Q_E"], "sys_inv": g(sc[sc["sys_costs"].str.startswith("inv_")]),
                      "sys_op": g(sc[sc["sys_costs"].str.startswith("op_")]),
                      "smr_new_MW": g(cn), "gas_price": g(rd(batch, "repgasprice_nat").query("case == @case")),
                      "gas_quads": g(rd(batch, "repgasquant_nat").query("case == @case")),
                      "gas_cost": g(rd(batch, "gascost_cendiv").query("case == @case")), "co2_t": g(em), "smr_MW": g(cap)})
    D["dual"] = g(du[du["case"] == case]) if du is not None else 0.0
    return D

annual_fc = lambda s: np.interp(YRS, s.index.values.astype(float), s.values.astype(float))
pv_fc = lambda s: float((annual_fc(s) * DISC).sum())
lump_fc = lambda s: float((s * FC_DR ** -(s.index.values.astype(float) - 2026)).sum())
Bf = fc_table(*BASE)
rows = []
for t in TARGETS:
    R = fc_table(*PAIRS[t][0]); d = R - Bf
    sup = R["dual"] * FC_PVF * R["smr_new_MW"]
    rows.append(dict(target=t, smr_GW_2050=R["smr_MW"].loc[2050] / 1e3, support_PV_B=lump_fc(sup) * FC_TO25 / 1e9,
                     d_energy_pay_PV_B=pv_fc(d["load"]) * FC_TO25 / 1e9, d_capacity_pay_PV_B=pv_fc(d["res_marg"]) * FC_TO25 / 1e9,
                     d_rps_pay_PV_B=pv_fc(d["state_rps"]) * FC_TO25 / 1e9,
                     d_elec_pay_PV_B=pv_fc(d["load"] + d["res_marg"] + d["state_rps"]) * FC_TO25 / 1e9,
                     d_sys_inv_PV_B=lump_fc(d["sys_inv"]) * FC_TO25 / 1e9, d_sys_op_PV_B=pv_fc(d["sys_op"]) * FC_TO25 / 1e9,
                     d_syscost_PV_B=(lump_fc(d["sys_inv"]) + pv_fc(d["sys_op"])) * FC_TO25 / 1e9,
                     d_gascost_PV_B=pv_fc(d["gas_cost"]) * FC_TO25 / 1e9,
                     gas_price_base_2050=Bf["gas_price"].loc[2050] * FC_TO25, d_gas_price_2050=d["gas_price"].loc[2050] * FC_TO25,
                     d_gas_quads_2050=d["gas_quads"].loc[2050], d_co2_Mt_2050=d["co2_t"].loc[2050] / 1e6,
                     d_co2_Mt_cum=float(annual_fc(d["co2_t"]).sum()) / 1e6,
                     load_price_base_2050=Bf["load"].loc[2050] / Bf["load_MWh"].loc[2050] * FC_TO25,
                     d_load_price_2050=(R["load"].loc[2050] / R["load_MWh"].loc[2050] - Bf["load"].loc[2050] / Bf["load_MWh"].loc[2050]) * FC_TO25,
                     d_load_MWh_2050=d["load_MWh"].loc[2050] / 1e6))
mine = pd.DataFrame(rows).set_index("target")
first = pd.read_csv(paths.ANALYSIS / "scratch" / "ag_firstcut_20261020" / "size_transfer_p50.csv").set_index("target")
C5 = (mine.stack().rename("reproduced").to_frame().join(first.stack().rename("first_cut")))
C5["diff"] = C5["reproduced"] - C5["first_cut"]
C5["rel_diff"] = C5["diff"] / C5["first_cut"].where(C5["first_cut"].abs() > 1e-9)
C5 = C5.reset_index().rename(columns={"level_1": "column"})
print("C5: max |relative difference| by column:")
print(C5.groupby("column")["rel_diff"].apply(lambda s: s.abs().max()).round(6).to_string())

C5: max |relative difference| by column:
column
d_capacity_pay_PV_B     0.000032
d_co2_Mt_2050           0.000000
d_co2_Mt_cum            0.000000
d_elec_pay_PV_B         0.000003
d_energy_pay_PV_B       0.000000
d_gas_price_2050        0.000000
d_gas_quads_2050        0.000000
d_gascost_PV_B          0.000000
d_load_MWh_2050              NaN
d_load_price_2050       0.000000
d_rps_pay_PV_B          0.000000
d_sys_inv_PV_B          0.000000
d_sys_op_PV_B           0.000000
d_syscost_PV_B          0.000000
gas_price_base_2050     0.000000
load_price_base_2050    0.000000
smr_GW_2050             0.000000
support_PV_B            0.000000


**C5, named differences.** Under the first cut's own time handling (flows 2026–2050 at DR 1.0536,
lumps whole), the two definitional changes of the AG ledger move these quantities:
(i) capacity payments without the `res_marg_ann` term, and (ii) system cost without the credit rows
and the ACP rows (the resource cost of view 2).

In [10]:
rows = []
for t in TARGETS:
    R = fc_table(*PAIRS[t][0]); d = R - Bf
    der_r = OUTS[t]["memo_run"]["der"]; der_b = OUTS[t]["memo_base"]["der"]
    cap_new = pv_fc(der_r["P_C"] - der_b["P_C"]) * FC_TO25 / 1e9
    acc = OUTS[t]["delta_accounts"]
    sel = lambda e: acc[(acc["actor"] == "system") & (acc["entry"] == e)].set_index("t")["Value"]
    res_cost = -(lump_fc(sel("resource cost: capital")) + pv_fc(sel("resource cost: operating"))) * FC_TO25 / 1e9
    rows.append(dict(target=t, capacity_firstcut=mine.loc[t, "d_capacity_pay_PV_B"], capacity_AG=cap_new,
                     res_marg_ann_term=mine.loc[t, "d_capacity_pay_PV_B"] - cap_new,
                     syscost_firstcut=mine.loc[t, "d_syscost_PV_B"], resource_cost_AG=res_cost,
                     credit_and_ACP_rows=mine.loc[t, "d_syscost_PV_B"] - res_cost))
C5N = pd.DataFrame(rows)
print(C5N.round(2).to_string(index=False))

target  capacity_firstcut  capacity_AG  res_marg_ann_term  syscost_firstcut  resource_cost_AG  credit_and_ACP_rows
   eia               1.97         1.58               0.40             46.31             44.41                 1.90
    aj             -16.27        -5.42             -10.85             66.79             64.31                 2.49
  iaea             -24.54        -9.00             -15.54            133.18            125.24                 7.94
   mck             -32.87       -12.62             -20.25            178.58            169.59                 8.99
 cop28             -93.17       -40.31             -52.86            297.63            286.15                11.48
    eo            -164.14       -71.95             -92.18            490.91            476.25                14.67


In [11]:
closure = pd.concat([CL.assign(table="C1/C2/RPS by year"), C3.assign(table="C3 by horizon and rate")], ignore_index=True)
closure.to_csv(OUT / "ag_closure.csv", index=False)
RPS.to_csv(OUT / "ag_closure_rps_terms.csv", index=False)
C4.to_csv(OUT / "ag_c4_support.csv", index=False)
C5.to_csv(OUT / "ag_c5_firstcut.csv", index=False)
C5N.to_csv(OUT / "ag_c5_named_differences.csv", index=False)
print("written:", [p.name for p in sorted(OUT.glob("ag_*.csv"))])

written: ['ag_actor_pv.csv', 'ag_annual_A.csv', 'ag_c4_support.csv', 'ag_c5_firstcut.csv', 'ag_c5_named_differences.csv', 'ag_closure.csv', 'ag_closure_rps_terms.csv', 'ag_emissions_delta.csv', 'ag_views.csv']


## 5. The ledger exports

Every actor entry as a PV at every rate, every view variant, the annual entries under horizon A and
the emissions changes. Money in 2025 $; PVs to 2026.

In [12]:
pv_rows, ann_rows, em_rows = [], [], []
for t, o in OUTS.items():
    pv_rows.append(L.actor_pvs(o, True).assign(target=t, run=PAIRS[t][0][1], baseline=BASE[1]))
    G = L.gain_tables(o, True)
    G = G[G["horizon"] == "A"].copy()
    G["Value"] = G["Value"] * TO25 / 1e9
    ann_rows.append(G.assign(target=t))
    e = o["A"].loc["emissions"]
    em_rows.append(e.stack().rename("tonnes").reset_index().rename(columns={"level_0": "entry", "level_1": "year"}).assign(target=t))
AP = pd.concat(pv_rows, ignore_index=True)
VW = pd.concat([v.assign(target=t) for t, v in VIEWS.items()], ignore_index=True)
AN = pd.concat(ann_rows, ignore_index=True).rename(columns={"Value": "gain_2025B"})
EM = pd.concat(em_rows, ignore_index=True)
AP.to_csv(OUT / "ag_actor_pv.csv", index=False)
VW.to_csv(OUT / "ag_views.csv", index=False)
AN.to_csv(OUT / "ag_annual_A.csv", index=False)
EM.to_csv(OUT / "ag_emissions_delta.csv", index=False)
print({k: len(v) for k, v in dict(actor_pv=AP, views=VW, annual_A=AN, emissions=EM).items()})

{'actor_pv': 17328, 'views': 38880, 'annual_A': 216600, 'emissions': 900}


## 6. Winners and losers, one display configuration

The table sums each actor's entries. It shows funding case T and the lump support form, under both
horizons, at 2.0% (a resident-view rate) and 5.36% (the developer rate), with allowances allocated free
(the treatment implicit in the ReEDS outputs; §6b shows the auctioned treatment). The environment has one entry
per carbon basis and per health variant, so it is never summed: the 2.0% table shows the global EPA 2023
2.0% value and AP2 ACS below the money rows, and the 5.36% table, which has no paired carbon value,
omits it. This is a display choice for the H2 review, not a headline: every other combination is in
`ag_actor_pv.csv`. Rows: actors; columns: cost targets; 2025 $B, PV to 2026; a positive value is a gain.

In [13]:
FREE = lambda d: d[d["allowance_state"].isin(["free", "none"]) & d["allowance_national"].isin(["free", "none"])]
sel = FREE(AP[(AP["funding"] == "T") & (AP["support_form"] == "lump")])
money = sel[(sel["actor"] != "environment") & ~sel["actor"].str.startswith("foreign")]
for h in ("A", "B"):
    for col in ("PV_2.0%", "PV_5.36%"):
        w = money[money["horizon"] == h].groupby(["actor", "target"])[col].sum().unstack("target")[TARGETS]
        w = w[(w.abs() > 0.05).any(axis=1)]
        w.loc["sum of US money entries"] = w.sum()
        f = sel[(sel["horizon"] == h) & sel["actor"].str.startswith("foreign")].groupby("target")[col].sum()
        w.loc["foreign buyers and sellers (outside the US ledger)"] = f.reindex(TARGETS).values
        if col == "PV_2.0%":
            e = sel[(sel["horizon"] == h) & sel["entry"].isin(["GHG: global EPA2023 2.0%", "health: AP2 ACS"])]
            e = e.pivot_table(index="entry", columns="target", values=col)[TARGETS]
            w = pd.concat([w, e.rename(index=lambda x: f"environment, {x}")])
        print(f"\nhorizon {h}, {col}:")
        print(w.round(1).to_string())


horizon A, PV_2.0%:
target                                               eia    aj   iaea    mck  cop28      eo
electricity consumers                               32.7  51.1  133.4  162.6  353.4   585.9
gas producers                                      -26.9 -26.2  -74.6  -99.0 -206.7  -294.0
network                                              0.6  -1.2   -5.4   -5.8  -12.9   -17.2
non-power gas consumers                             10.8  10.4   29.4   39.3   84.3   121.9
owners: H2 and DAC production                       -0.0  -0.5   -0.5   -0.2   -0.2    -0.1
owners: SMR (new)                                    6.6   8.1    8.2   14.7    9.1     9.0
owners: coal                                        -0.7  -0.2   -2.6   -2.8   -6.7   -10.6
owners: existing nuclear                            -4.2  -5.3  -16.1  -19.4  -43.5   -70.6
owners: gas                                         -3.2 -11.5  -18.6  -25.7  -67.7  -122.1
owners: hydro                                       -3.1  -

**Environment entries** (gain = minus the change in damages), horizon A at the paired rate of each
carbon value, and the six health variants at 2.0%.

In [14]:
env = FREE(AP[(AP["actor"] == "environment") & (AP["funding"] == "T") & (AP["support_form"] == "lump") & (AP["horizon"] == "A")])
rows = {}
for _, r in env.iterrows():
    rk = next((L.SCC_PAIRS[b] for b in L.SCC_PAIRS if b in r["entry"]), "2.0%")
    rows[(r["entry"], r["target"])] = r[f"PV_{rk}"]
print(pd.Series(rows).unstack()[TARGETS].round(1).to_string())
cum = EM.groupby(["target", "entry"])["tonnes"].sum().unstack("target")[TARGETS] / 1e6
print("\ncumulative emissions change 2026-2050 (Mt):")
print(cum.round(1).to_string())

                           eia    aj   iaea    mck   cop28      eo
GHG: domestic EPA2019 3%   1.2   1.2    5.8    8.2    18.8    33.7
GHG: domestic EPA2019 7%   0.1   0.1    0.6    0.9     1.9     3.5
GHG: global EPA2023 1.5%  74.1  73.4  337.6  478.2  1101.6  1970.5
GHG: global EPA2023 2.0%  43.4  43.0  197.6  279.3   641.6  1147.2
GHG: global EPA2023 2.5%  26.6  26.5  121.5  171.1   391.9   700.4
health: AP2 ACS            0.4   0.9    2.8    3.6     7.1    12.8
health: AP2 H6C            1.1   2.2    7.1    9.3    18.3    33.0
health: EASIUR ACS         0.9   1.5    4.6    5.8    10.0    17.1
health: EASIUR H6C         2.3   3.8   11.9   14.9    25.7    43.9
health: InMAP ACS          0.6   1.2    3.9    5.0     9.8    18.0
health: InMAP H6C          1.5   3.2    9.9   12.8    25.2    46.1

cumulative emissions change 2026-2050 (Mt):
target          eia     aj   iaea    mck   cop28      eo
entry                                                   
CH4 process    -0.0   -0.0   -0.0   -

**Resident views, reference variant** (funding T, lump support, AP2 ACS, MCPF 1, φ = 0) at each
carbon basis and its paired rate. All other variants are in `ag_views.csv`.

In [15]:
ref = FREE(VW[(VW["funding"] == "T") & (VW["support_form"] == "lump") & (VW["health_model"] == "AP2 ACS")
         & (VW["mcpf"] == 1.0) & (VW["phi"] == 0.0)])
cols = ["view1", "view2", "profits_domestic", "minus_d_resource_cost", "env_ghg", "env_health", "foreign_gain", "C3_residual"]
for h in ("A", "B"):
    print(f"\nhorizon {h}:")
    print(ref[ref["horizon"] == h].set_index(["scc_basis", "target"])[cols].round(1).to_string())


horizon A:


                             view1   view2  profits_domestic  minus_d_resource_cost  env_ghg  env_health  foreign_gain  C3_residual
scc_basis           target                                                                                                         
global EPA2023 1.5% eia       71.6    22.1             -49.7                  -45.4     74.1         0.5           7.1          0.3
global EPA2023 2.0% eia       41.7    -4.2             -46.2                  -41.4     43.4         0.4           6.6          0.2
global EPA2023 2.5% eia       25.7   -17.0             -42.9                  -37.9     26.6         0.4           6.1          0.2
domestic EPA2019 3% eia        1.0   -38.7             -39.9                  -34.6      1.2         0.4           5.7          0.2
domestic EPA2019 7% eia        2.3   -20.3             -22.7                  -17.4      0.1         0.2           3.2          0.1
global EPA2023 1.5% aj        98.6    25.9             -71.6                

## 6b. Allowance treatments for the state carbon programmes (amendment A14)

ReEDS prices the state caps (CA; DE and NJ as modelled) and RGGI as constraints, so the outputs leave
the allowance value with emitters (`free`). Under `auctioned`, emitters pay price × covered emissions,
the network pays the import part of a state cap, and state governments (inside taxpayers) receive the
total. The stage-1 runs have no national cap, so `allowance_national` is `none`. The table gives each
actor's total under both treatments and the difference (auctioned − free); horizon A, 2.0%, funding T,
lump support; 2025 $B, PV to 2026.

In [16]:
sel = AP[(AP["funding"] == "T") & (AP["support_form"] == "lump") & (AP["horizon"] == "A")
         & (AP["actor"] != "environment")]
tot = sel.groupby(["allowance_state", "actor", "target"])["PV_2.0%"].sum().unstack("target")[TARGETS]
dif = (tot.loc["auctioned"] - tot.loc["free"]).fillna(tot.loc["auctioned"])
print("difference, auctioned - free:")
print(dif[(dif.abs() > 0.005).any(axis=1)].round(2).to_string())
rec = sel[(sel["allowance_state"] == "auctioned") & (sel["entry"] == L.ALLOW_ENTRY["state"])]
print("\nstate allowance value, policy run minus baseline (taxpayers' receipts under auctioned):")
print(rec.groupby(["actor", "target"])["PV_2.0%"].sum().unstack("target")[TARGETS].round(2).to_string())
vv = VW[(VW["funding"] == "T") & (VW["support_form"] == "lump") & (VW["health_model"] == "AP2 ACS") & (VW["phi"] == 0.0)
        & (VW["horizon"] == "A") & (VW["scc_basis"] == "global EPA2023 2.0%")]
w = vv.pivot_table(index=["mcpf", "target"], columns="allowance_state", values=["view1", "view2"])
for c in ("view1", "view2"):
    w[(c, "auctioned - free")] = w[(c, "auctioned")] - w[(c, "free")]
print("\nviews by treatment (horizon A, global EPA 2023 2.0%, AP2 ACS, phi = 0):")
print(w.round(2).to_string())
print("max |C3| over all variants (2025 $B):", float(VW["C3_residual"].abs().max()))

difference, auctioned - free:
target          eia    aj  iaea   mck  cop28     eo
actor                                              
network        0.70  2.04  2.94  3.05   5.24   8.78
owners: gas    0.91  1.91  1.85  1.84   3.35   6.73
owners: other -0.79 -1.77 -2.28 -2.40  -3.45  -3.87
taxpayers     -0.82 -2.18 -2.52 -2.49  -5.14 -11.65

state allowance value, policy run minus baseline (taxpayers' receipts under auctioned):
target                              eia    aj  iaea   mck  cop28     eo
actor                                                                  
foreign sellers: Canadian imports  0.00  0.00  0.00  0.00   0.00   0.00
network                            0.70  2.04  2.94  3.05   5.24   8.78
owners: H2 and DAC production      0.00  0.00  0.00  0.00   0.00   0.00
owners: SMR (new)                  0.00  0.00  0.00  0.00   0.00   0.00
owners: coal                       0.00  0.00  0.00 -0.00   0.00   0.00
owners: existing nuclear           0.00  0.00  0.00  0.00   0.00 

                    view1                       view2                            view1            view2
allowance_state auctioned     free   policy auctioned    free  policy auctioned - free auctioned - free
mcpf target                                                                                            
1.00 aj             63.82    66.00    65.42     -0.22   -0.22   -0.22            -2.18              0.0
     cop28         775.24   780.39   779.22    313.86  313.86  313.86            -5.14              0.0
     eia            40.89    41.71    41.60     -4.23   -4.23   -4.23            -0.82              0.0
     eo           1415.96  1427.61  1420.07    710.11  710.11  710.11           -11.65              0.0
     iaea          203.88   206.39   206.74     34.23   34.23   34.23            -2.52              0.0
     mck           310.52   313.01   313.29     98.27   98.27   98.27            -2.49              0.0
1.25 aj             53.96    56.13    55.55    -10.08  -10.08  -

**Allowance checks.** In each run, the process CO2 by technology and zone (`emit_irt`) must sum to
the national process CO2 (`emit_nat`). The import part of each state cap and the RGGI slack
(price × (cap − covered emissions)) are listed; the RGGI slack should be zero where the price is positive.

In [17]:
rows = []
for t, o in OUTS.items():
    for who in ("memo_run", "memo_base"):
        m = o[who]
        rows.append(dict(target=t, who=who[5:], emit_irt_minus_emit_nat_Mt=float((m["co2_process_irt"] - m["co2_process_nat"]).abs().max() / 1e6),
                         state_import_part_B_max=float(m["state_import_part"].max() / 1e9),
                         state_import_part_B_min=float(m["state_import_part"].min() / 1e9),
                         rggi_slack_B_maxabs=float(m["rggi_slack"].abs().max() / 1e9)))
print(pd.DataFrame(rows).round(4).to_string(index=False))

target  who  emit_irt_minus_emit_nat_Mt  state_import_part_B_max  state_import_part_B_min  rggi_slack_B_maxabs
   eia  run                      0.0001                   1.7278                      0.0                  0.0
   eia base                      0.0001                   1.7345                      0.0                  0.0
    aj  run                      0.0001                   1.5527                      0.0                  0.0
    aj base                      0.0001                   1.7345                      0.0                  0.0
  iaea  run                      0.0000                   1.5022                      0.0                  0.0
  iaea base                      0.0001                   1.7345                      0.0                  0.0
   mck  run                      0.0000                   1.4975                      0.0                  0.0
   mck base                      0.0001                   1.7345                      0.0                  0.0
 

## 7. What reads these exports

The H2 review (Ethan) reads `ag_closure.csv` and the winners-and-losers tables. Notebook 22 (baseline
band) reuses `aglib.ledger.pair` against each of the 19 baselines. Observations go to
`docs/observations.md` by hand, with no interpretation (protocol §10).